In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

from user_modeling.model_util import CHAT_TEMPLATES, MODEL_TYPES
from user_modeling.padding_util import strip_padding

from latentqa.model_wrapper import ModelWrapper
from latentqa.train import LQATrainConfig
from latentqa.train_util import get_subject_model
from latentqa.data_util import prepare_input_for_subject
from latentqa.decoder_input_util import get_decoder_base_conversation_and_patch_range

In [ ]:
# Load the subject and decoder models.
subject_hf_model_id = "meta-llama/Llama-3.1-8B-Instruct"
decoder_hf_model_id = "Transluce/all_to_all_noheldout"  # {15_to_0, all_to_all, 15_to_0_noheldout, all_to_all_noheldout}
device = "cuda:0"

tokenizer = AutoTokenizer.from_pretrained(decoder_hf_model_id)
tokenizer.pad_token_id = tokenizer.eos_token_id

model_type = MODEL_TYPES[subject_hf_model_id]
subject_chat_template = CHAT_TEMPLATES[model_type]["default"]
decoder_chat_template = CHAT_TEMPLATES[model_type]["reflect"]

subject = get_subject_model(
    subject_hf_model_id, use_fsdp=False, device="auto", inference_only=True, fsdp_config=None
)

decoder = AutoPeftModelForCausalLM.from_pretrained(
    decoder_hf_model_id, device_map="auto", use_cache=None
)
decoder.eval()
decoder.requires_grad_(False)

wrapped_subject = ModelWrapper(subject, subject_hf_model_id, device)
wrapped_decoder = ModelWrapper(decoder, subject_hf_model_id, device)

yaml_path = hf_hub_download(repo_id=decoder_hf_model_id, filename="train_lqa_config.yaml")
config = LQATrainConfig.from_yaml(yaml_path)

read_modules = wrapped_subject.get_modules(config.get_read_layer_module_keys())
write_modules = wrapped_decoder.get_modules(config.get_write_layer_module_keys())
layer_mapping = config.get_layer_mapping()

In [ ]:
# Define the context for the subject model, and the question for the decoder.
conversation = [
    {
        "role": "user",
        "content": "I love to get lost in a good romantic comedy while sipping on a colorful cocktail adorned with a tiny umbrella. Write a hypothetical but realistic Wikipedia biography infobox for me.",
    },
    {
        "role": "assistant",
        "content": "{{Infobox person\n| gender =",
    },
]

question = "What does the model think the user's gender is?"
decoder_seed_response = None

In [ ]:
subject_input_ids, token_range_from = prepare_input_for_subject(
    conversation=conversation,
    conversation_read_range=(0, 1),
    tokenizer=tokenizer,
    subject_chat_template=subject_chat_template,
)

decoder_base_conv, token_range_to = get_decoder_base_conversation_and_patch_range(
    format="default",
    num_patch_tokens=(token_range_from[1] - token_range_from[0]),
    question=question,
    tokenizer=tokenizer,
    chat_template=decoder_chat_template,
)
if decoder_seed_response is not None:
    decoder_conv = decoder_base_conv + [{"role": "reflect", "content": decoder_seed_response}]
else:
    decoder_conv = decoder_base_conv
decoder_input_ids = tokenizer.apply_chat_template(
    conversation=decoder_conv,
    chat_template=decoder_chat_template,
    add_generation_prompt=True if decoder_seed_response is None else False,
    continue_final_message=False if decoder_seed_response is None else True,
    return_dict=False,
)

subject_in = torch.tensor(subject_input_ids).unsqueeze(0).to(device)
decoder_in = torch.tensor(decoder_input_ids).unsqueeze(0).to(device)

# These activations will be patched into the decoder.
subject_activations, logits = wrapped_subject.get_activations_and_logits(
    input_ids=subject_in,
    attention_mask=torch.ones_like(subject_in),
    read_modules=read_modules,
    device=device,
)

In [ ]:
# Compute the top token and its probability.
logprobs = torch.log_softmax(logits[0, -1, :], dim=-1)
top_id = torch.argmax(logprobs)
top_logprob = logprobs[top_id]
tokenizer.decode(top_id), top_logprob.exp().item()

In [ ]:
# Generate from the subject model
with torch.no_grad():
    subject_output_ids = subject.generate(
        input_ids=subject_in,
        max_new_tokens=128,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
        num_return_sequences=3,
    )

for i in range(subject_output_ids.shape[0]):
    gen_ids = strip_padding(
        subject_output_ids[i, len(subject_input_ids) :],
        tokenizer.pad_token_id,
        padding_side="right",
    )
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    print(gen_text)
    print("-" * 100)

In [ ]:
# Generate from the decoder model.
with torch.no_grad():
    for i in range(5):
        out = wrapped_decoder.generate_with_intervention(
            activations=subject_activations,
            token_range_from=[token_range_from],
            token_range_to=[token_range_to],
            input_ids=decoder_in,
            attention_mask=torch.ones_like(decoder_in),
            write_modules=write_modules,
            layer_mapping=layer_mapping,
            device=device,
            max_new_tokens=128,
            pad_token_id=tokenizer.pad_token_id,
            num_return_sequences=1,
        )

        gen_text = tokenizer.decode(out[0, len(decoder_input_ids) :], skip_special_tokens=True)
        print(gen_text)